In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the slightly simpler NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 9  # Reduced embedding dimension by 10%

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers with slightly reduced complexity and regularization
dense1 = keras.layers.Dense(115, activation='relu')(concat)  # Reduced units by ~10%
dropout1 = keras.layers.Dropout(0.2)(dense1)
dense2 = keras.layers.Dense(58, activation='relu')(dropout1)  # Reduced units by ~10%
dropout2 = keras.layers.Dropout(0.2)(dense2)
output = keras.layers.Dense(1)(dropout2)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=90, learning_rate=0.1, max_depth=6)  # Slightly reduced n_estimators
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Normalize the predicted ratings to a range between 0 and 100
scaler = MinMaxScaler(feature_range=(0, 100))
xgb_predictions_comb_normalized = scaler.fit_transform(xgb_predictions_comb.reshape(-1, 1)).flatten()

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb_normalized),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb_normalized
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
import multiprocessing

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the slightly simpler NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 9  # Reduced embedding dimension by 10%

# Use MirroredStrategy for parallel training
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    # Client embedding
    client_input = keras.layers.Input(shape=(1,), name='client_input')
    client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
    client_vec = keras.layers.Flatten()(client_embedding)

    # Product embedding
    product_input = keras.layers.Input(shape=(1,), name='product_input')
    product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
    product_vec = keras.layers.Flatten()(product_embedding)

    # Concatenate embeddings
    concat = keras.layers.Concatenate()([client_vec, product_vec])

    # Fully connected layers with slightly reduced complexity and regularization
    dense1 = keras.layers.Dense(115, activation='relu')(concat)  # Reduced units by ~10%
    dropout1 = keras.layers.Dropout(0.2)(dense1)
    dense2 = keras.layers.Dense(58, activation='relu')(dropout1)  # Reduced units by ~10%
    dropout2 = keras.layers.Dropout(0.2)(dense2)
    output = keras.layers.Dense(1)(dropout2)

    # Define the model
    model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Determine the number of CPU cores
num_cores = multiprocessing.cpu_count()

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=90, learning_rate=0.1, max_depth=6, n_jobs=num_cores)  # Slightly reduced n_estimators
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Normalize the predicted ratings to a range between 0 and 100
scaler = MinMaxScaler(feature_range=(0, 100))
xgb_predictions_comb_normalized = scaler.fit_transform(xgb_predictions_comb.reshape(-1, 1)).flatten()

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb_normalized),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb_normalized
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
import multiprocessing

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the slightly simpler NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 9  # Reduced embedding dimension by 10%

# Use MirroredStrategy for parallel training
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    # Client embedding
    client_input = keras.layers.Input(shape=(1,), name='client_input')
    client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
    client_vec = keras.layers.Flatten()(client_embedding)

    # Product embedding
    product_input = keras.layers.Input(shape=(1,), name='product_input')
    product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
    product_vec = keras.layers.Flatten()(product_embedding)

    # Concatenate embeddings
    concat = keras.layers.Concatenate()([client_vec, product_vec])

    # Fully connected layers with slightly reduced complexity and regularization
    dense1 = keras.layers.Dense(115, activation='relu')(concat)  # Reduced units by ~10%
    dropout1 = keras.layers.Dropout(0.2)(dense1)
    dense2 = keras.layers.Dense(58, activation='relu')(dropout1)  # Reduced units by ~10%
    dropout2 = keras.layers.Dropout(0.2)(dense2)
    output = keras.layers.Dense(1)(dropout2)

    # Define the model
    model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')

# Convert data to tf.data.Dataset
def df_to_dataset(X, y, batch_size=64):
    dataset = tf.data.Dataset.from_tensor_slices((dict(X), y))
    dataset = dataset.shuffle(buffer_size=len(X))
    dataset = dataset.batch(batch_size)
    return dataset

# Create tf.data.Dataset for training
train_dataset = df_to_dataset(X_train, y_train)
validation_split = int(0.2 * len(X_train))
train_data = train_dataset.skip(validation_split)
val_data = train_dataset.take(validation_split)

# Train the model
history = model.fit(train_data, validation_data=val_data, epochs=5)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict(dict(X_train))
predictions_train = model.predict(dict(X_train))

# Determine the number of CPU cores
num_cores = multiprocessing.cpu_count()

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=90, learning_rate=0.1, max_depth=6, n_jobs=num_cores)  # Slightly reduced n_estimators
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict(dict(X_test))
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict({'client_input': all_client_ids, 'product_input': all_product_ids})
nn_predictions = model.predict({'client_input': all_client_ids, 'product_input': all_product_ids}).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Normalize the predicted ratings to a range between 0 and 100
scaler = MinMaxScaler(feature_range=(0, 100))
xgb_predictions_comb_normalized = scaler.fit_transform(xgb_predictions_comb.reshape(-1, 1)).flatten()

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb_normalized),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb_normalized
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
from joblib import Parallel, delayed

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the slightly simpler NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 9  # Reduced embedding dimension by 10%

# Use MirroredStrategy for parallel training
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    # Client embedding
    client_input = keras.layers.Input(shape=(1,), name='client_input')
    client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
    client_vec = keras.layers.Flatten()(client_embedding)

    # Product embedding
    product_input = keras.layers.Input(shape=(1,), name='product_input')
    product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
    product_vec = keras.layers.Flatten()(product_embedding)

    # Concatenate embeddings
    concat = keras.layers.Concatenate()([client_vec, product_vec])

    # Fully connected layers with slightly reduced complexity and regularization
    dense1 = keras.layers.Dense(115, activation='relu')(concat)  # Reduced units by ~10%
    dropout1 = keras.layers.Dropout(0.2)(dense1)
    dense2 = keras.layers.Dense(58, activation='relu')(dropout1)  # Reduced units by ~10%
    dropout2 = keras.layers.Dropout(0.2)(dense2)
    output = keras.layers.Dense(1)(dropout2)

    # Define the model
    model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
    model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Define a function to train the XGBoost model in parallel
def train_xgb_model(X, y):
    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=90, learning_rate=0.1, max_depth=6)  # Slightly reduced n_estimators
    xgb_model.fit(X, y)
    return xgb_model

# Train the XGBoost model in parallel
xgb_model = Parallel(n_jobs=-1)(delayed(train_xgb_model)(embeddings_train, y_train))[0]

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Normalize the predicted ratings to a range between 0 and 100
scaler = MinMaxScaler(feature_range=(0, 100))
xgb_predictions_comb_normalized = scaler.fit_transform(xgb_predictions_comb.reshape(-1, 1)).flatten()

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb_normalized),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb_normalized
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)
